# Sahayak — Fine-tune Gemma 4 **E2B** (QLoRA) on Kaggle **GPU T4 ×2** — hardened

End-to-end: install → GPU guard → HF auth **+ access preflight** → dataset auto-discover + validate
(hard gate) → **E2B QLoRA** train (**auto OOM retry**) → merged-weights export → upload to a private
HF repo. Companion to `docs/SAHAYAK_DATASET_SPEC.md` and `backend/finetune/`.
**Pure `transformers` + `peft` + `bitsandbytes` — no Unsloth, no TRL.** (The old Unsloth path
dtype-crashed on T4s — `float != c10::Half` in `per_layer_model_projection` — and was removed.)

**Hardened against the common Kaggle failure modes:**
- the install is **import-verified in a fresh interpreter**, with a forced clean reinstall fallback;
- only **GPU 0 is visible** to the whole pipeline (the trainer is single-GPU anyway);
- token validity, **gated-model access (Gemma license!), and transformers architecture support are
  all checked in Cell 3** — never 30 minutes into a run;
- the repo clone **retries, pulls pushed fixes on re-runs**, and **falls back to scripts uploaded
  under `/kaggle/input`**;
- the dataset is auto-discovered under `/kaggle/input` **or from the repo's own `backend/finetune/data/`**
  (so it works with no dataset attached at all); a **val split is auto-created** if only a train
  file exists;
- training **auto-retries CUDA OOM** with a smaller sequence length (identical effective batch 8);
- a **merge-export failure never invalidates the run** — adapters are saved first and the cell
  detects and tolerates it (the merged export is also auto-skipped when disk is too low);
- progress state persists in `/kaggle/working/sahayak_state.json` — after a kernel restart, re-run
  Cells 1–3 and jump straight to the cell you need;
- the upload **retries**, drops the occasionally-flaky `hf_transfer` fast path on failure, and
  excludes re-derivable checkpoints.

### Before you run
1. **Settings → Accelerator = `GPU T4 x2`** (never P100 — compute capability 6.0 lacks the
   bitsandbytes 4-bit kernels; T4 = 7.5). **Internet = On**.
2. **Add-ons → Secrets → add `HF_TOKEN`** (tick *attach to this notebook*) = a Hugging Face token
   whose account has **accepted the Gemma license** at
   [huggingface.co/google/gemma-4-E2B-it](https://huggingface.co/google/gemma-4-E2B-it).
   Use a **WRITE** token if you want Cell 8's upload to work. *Never paste the token in a cell.*
3. *(Optional)* **Add Input → your dataset** with `train.jsonl`, `val.jsonl`, `eval_holdout.jsonl`.
   Without it, Cell 5 falls back to the repo's committed `backend/finetune/data/`.

The script is single-GPU; the **second T4 idles** (harmless). Precision on a T4: Gemma 4 has
fp16-unsafe ops and the T4 has no native bf16, so the trainer runs **4-bit NF4 weights + float32
compute** — slower than bf16 but numerically safe. E2B fits this path on one 16 GB T4; use E4B only
on a native-bf16 GPU (Colab L4/A100), where the trainer switches to bf16 automatically.


## Cell 1 — install the fine-tune stack (verified)

Kaggle's image already ships torch + CUDA; we add the pinned Apache-2.0 stack from the repo's
`backend/finetune/pyproject.toml`: `transformers` (≥ 5.6 for native Gemma 4), `datasets`, `peft`,
`accelerate`, `bitsandbytes`. **No Unsloth, no TRL.**

After installing, the whole stack is **imported in a fresh interpreter** to prove it actually works;
on failure the cell automatically force-reinstalls clean and re-verifies.

> pip *warnings* about Kaggle's preinstalled RAPIDS/`cudf` packages are normal and harmless.
> Only if the final assert fires: **Run → Restart & clear cell outputs**, then re-run from Cell 1.


In [19]:
import os, subprocess, sys, time

# The trainer is single-GPU; hiding GPU 1 keeps every library honest about that.
# Must be set before ANY torch/CUDA import in this kernel -> first cell.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

# Same pins as backend/finetune/pyproject.toml (the repo isn't cloned yet at this point).
PINS = ['transformers>=5.6,<6', 'datasets>=3.6,<6', 'peft>=0.16,<1',
        'accelerate>=1.6,<2', 'bitsandbytes>=0.46,<1']

def pip(*args, retries=3):
    """pip with retries (Kaggle egress can be flaky); prints the log tail only on failure."""
    for attempt in range(1, retries + 1):
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', *args],
                           capture_output=True, text=True)
        if r.returncode == 0:
            return True
        print(f"[pip attempt {attempt}/{retries} FAILED] pip install {' '.join(args)}")
        print((r.stdout + r.stderr)[-2500:])
        time.sleep(5 * attempt)
    return False

def stack_imports_ok():
    """Import the whole stack in a FRESH interpreter: a broken half-upgrade is caught here,
    not 40 minutes into training, and this kernel stays free of stale modules."""
    r = subprocess.run(
        [sys.executable, '-c',
         'import torch, transformers, datasets, peft, accelerate, bitsandbytes;'
         'print("torch", torch.__version__, "| transformers", transformers.__version__,'
         '"| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)'],
        capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip()[-2000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
    return r.returncode == 0

ok = pip('-q', *PINS)
pip('-q', '-U', 'huggingface_hub[hf_transfer]', 'sentencepiece', 'protobuf')

if not (ok and stack_imports_ok()):
    print('\n[recover] standard install is broken -> clean forced reinstall ...')
    pip('-q', '--upgrade', '--force-reinstall', '--no-cache-dir', *PINS)
    assert stack_imports_ok(), (
        'Install is still broken. Do Run -> Restart & clear cell outputs, then re-run from '
        'Cell 1. Also confirm Settings -> Internet = On.')

print('\ninstall complete and import-verified')


torch 2.10.0+cu128 | transformers 5.13.1 | peft 0.19.1 | bitsandbytes 0.49.2

install complete and import-verified


## Cell 2 — GPU guard (reject P100, confirm T4)

Fails loudly *before* a long run if the accelerator is wrong. bitsandbytes 4-bit needs CUDA compute
capability ≥ 7.0. Only GPU 0 is used; the second T4 idles.


In [20]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')  # before torch import (kernel-restart safety)
import subprocess
import torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

assert torch.cuda.is_available(), \
    "No CUDA GPU visible. Settings -> Accelerator -> 'GPU T4 x2' and Internet = On, then Restart."
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}   compute capability: {major}.{minor}   VRAM: {vram:.1f} GB')
assert major >= 7, (
    f'Compute capability {major}.{minor} < 7.0 (e.g. Tesla P100 = 6.0). bitsandbytes 4-bit '
    "needs >= 7.0. Switch the accelerator to 'GPU T4 x2' (T4 = 7.5)."
)
if vram < 14:
    print(f'[warn] {vram:.1f} GB VRAM is tight for float32 compute — expect the OOM auto-retry '
          'ladder in Cell 6 to kick in.')
print('GPU OK for QLoRA (training uses GPU 0; the second T4 idles).')


Sat Jul 11 18:42:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 3 — Hugging Face auth + access preflight

The token is read from Kaggle **Secrets** (`HF_TOKEN`, with `HUGGINGFACE_TOKEN`/`HF_API_TOKEN`
accepted as fallback names) — it is never written into the notebook or any file. The cell then
verifies **now**, instead of mid-run: the token is valid, the **gated** `google/gemma-4-E2B-it`
repo is actually accessible (i.e. the Gemma license was accepted by the token's account), and the
installed `transformers` supports the architecture.


In [21]:
import json, os

_STATE = '/kaggle/working/sahayak_state.json'

def state_load():
    return json.load(open(_STATE)) if os.path.exists(_STATE) else {}

def state_save(**kw):
    s = state_load(); s.update(kw)
    with open(_STATE, 'w') as f:
        json.dump(s, f, indent=1)

BASE_MODEL = 'google/gemma-4-E2B-it'

token = (os.environ.get('HF_TOKEN') or '').strip()
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in ('HF_TOKEN', 'HUGGINGFACE_TOKEN', 'HF_API_TOKEN'):
            try:
                token = (client.get_secret(name) or '').strip()
                if token:
                    print(f'token loaded from Kaggle secret {name!r}')
                    break
            except Exception:
                pass
    except Exception as e:
        print('kaggle_secrets unavailable:', repr(e))

if not token:
    raise SystemExit(
        'No Hugging Face token found. Add-ons -> Secrets -> add HF_TOKEN (tick "attach to this '
        'notebook"!), using a token whose HF account has accepted the Gemma license at '
        'huggingface.co/' + BASE_MODEL + ' — then re-run this cell.')

os.environ['HF_TOKEN'] = token
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Fail NOW (not 30 min into a run) on: bad token / gated model / transformers too old.
from huggingface_hub import HfApi
api = HfApi(token=token)
try:
    me = api.whoami()
except Exception as e:
    raise SystemExit('HF token is invalid or expired: ' + repr(e)
                     + '\nCreate a new token at huggingface.co/settings/tokens and update the '
                       'Kaggle secret, then re-run this cell.')
print('authenticated as:', me.get('name'))
role = (me.get('auth') or {}).get('accessToken', {}).get('role')
if role == 'read':
    print('[warn] token is READ-only: training works, but Cell 8 (upload) needs a WRITE token.')

try:
    api.model_info(BASE_MODEL)
    print('model repo reachable:', BASE_MODEL)
except Exception as e:
    raise SystemExit(
        f'cannot access {BASE_MODEL}: {e!r}\n'
        'Gemma is GATED: open huggingface.co/' + BASE_MODEL + ' logged in as the token '
        'account, accept the license, then re-run this cell.')

try:
    from transformers import AutoConfig
    AutoConfig.from_pretrained(BASE_MODEL, token=token)
    print('architecture is supported by the installed transformers.')
except Exception as e:
    raise SystemExit(
        'The installed transformers cannot load this model config (usually: transformers too '
        'old for the architecture, or the license gate above). Re-run Cell 1; if it upgraded '
        'anything, Restart and run from Cell 1 again. Error: ' + repr(e))

state_save(BASE_MODEL=BASE_MODEL)
print('auth + preflight OK')


authenticated as: kesav2k04
model repo reachable: google/gemma-4-E2B-it


config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

architecture is supported by the installed transformers.
auth + preflight OK


## Cell 4 — get the training code (clone with fallback)

Clones the repo for `sahayak_finetune.py` + `validate_dataset.py` and `cd`s into `backend/finetune/` (so
the validator can import the canonical `SYSTEM_PROMPT` from the trainer). **Re-runs in the same
session `git pull` first**, so pushed fixes are picked up instead of training stale code. The clone
**retries 3×**; if it still fails (private repo / GitHub outage), the cell **falls back to any
`sahayak_finetune.py` found under `/kaggle/input`** — upload the `backend/finetune/` folder's `.py` files
as a Kaggle dataset and you never depend on GitHub at all.


In [22]:
import json, os

_STATE = '/kaggle/working/sahayak_state.json'

def state_load():
    return json.load(open(_STATE)) if os.path.exists(_STATE) else {}

def state_save(**kw):
    s = state_load(); s.update(kw)
    with open(_STATE, 'w') as f:
        json.dump(s, f, indent=1)
import glob, shutil, subprocess, time

REPO_URL = 'https://github.com/SivaNithishKumar/sankat-mochan.git'
WORK = '/kaggle/working'
repo_dir = os.path.join(WORK, 'sankat-mochan')
FT = None

def has_trainer(d):
    return bool(d) and os.path.exists(os.path.join(d, 'sahayak_finetune.py'))

# 1) already cloned earlier in this session -> pull pushed fixes instead of training stale code
if has_trainer(os.path.join(repo_dir, 'backend', 'finetune')):
    rc = subprocess.run(['git', '-C', repo_dir, 'pull', '--ff-only']).returncode
    if rc != 0:
        print('[warn] git pull failed — continuing with the already-cloned copy.')
    FT = os.path.join(repo_dir, 'backend', 'finetune')

# 2) clone, with retries
if FT is None:
    for attempt in range(1, 4):
        shutil.rmtree(repo_dir, ignore_errors=True)
        rc = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, repo_dir]).returncode
        if rc == 0 and has_trainer(os.path.join(repo_dir, 'backend', 'finetune')):
            FT = os.path.join(repo_dir, 'backend', 'finetune')
            break
        print(f'[warn] clone attempt {attempt}/3 failed; retrying ...')
        time.sleep(5)

# 3) fallback: the scripts were uploaded as (part of) a Kaggle dataset
if FT is None:
    hits = sorted(glob.glob('/kaggle/input/**/sahayak_finetune.py', recursive=True))
    if hits:
        src = os.path.dirname(hits[0])
        FT = os.path.join(WORK, 'finetune_code')
        os.makedirs(FT, exist_ok=True)
        for fn in os.listdir(src):
            if fn.endswith('.py'):
                shutil.copy2(os.path.join(src, fn), os.path.join(FT, fn))
        print('GitHub unavailable — using scripts found in Kaggle input:', src)

assert has_trainer(FT), (
    'Could not get sahayak_finetune.py. Either make the GitHub repo public, or upload the '
    "repo's finetune/ folder (its .py files) as a Kaggle dataset (Add Input) and re-run this cell.")

if not os.path.exists(os.path.join(FT, 'validate_dataset.py')):
    print('[warn] validate_dataset.py missing — Cell 5 will skip the mechanical validation gate.')

os.chdir(FT)
state_save(FT=FT)
print('finetune dir:', FT)


From https://github.com/SivaNithishKumar/sankat-mochan
   6774f4b..47cb8e0  main       -> origin/main


Updating 6774f4b..47cb8e0
Fast-forward
 .gitignore                                         |   4 +
 backend/.gitignore                            |   2 +
 backend/app.py                                |  10 +
 backend/autodetect_bench.py                   | 181 +++++
 backend/dev.sh                                | 108 +++
 backend/lid_test.py                           |  83 ++
 backend/requirements.txt                      |  11 +
 backend/stt.py                                | 135 +++-
 .../{index-G41SHFfQ.js => index-fNngBnk8.js}       |   2 +-
 backend/web/dist/index.html                   |   2 +-
 backend/web/src/components/VoiceRecorder.jsx  |  84 +-
 backend/web/src/lib/mapConfig.js              |  15 +-
 dev.ps1                                            |   8 +-
 docs/AI-Hub-Compile-Guide.md                       | 149 ++++
 docs/training plan.md                              | 219 ++++++
 finetune/kaggle_gemma4_e2b_finetune.ipynb          | 112 +--
 backend/finetune/pyprojec

## Cell 5 — locate the dataset & validate (HARD GATE)

Auto-finds the JSONL from **either** source, in this priority order:
1. an attached Kaggle dataset, anywhere under `/kaggle/input` (exact names → filename keywords
   `train`/`val`/`hold` → the only other JSONL present);
2. **the repo's own `backend/finetune/data/` folder** that Cell 4 cloned — so training works even with
   *no* dataset attached at all.

If no `val.jsonl` exists, a deterministic **~5% val split is carved off the training file
automatically** (training still gets an eval signal). You can also pin paths explicitly via the
`*_OVERRIDE` knobs. If nothing is found, the cell prints the **complete file listing of
`/kaggle/input`** so you can see exactly what is (or isn't) attached.

The mechanical validator (`SAHAYAK_DATASET_SPEC.md` §8.1) is a **hard gate** — training never runs
on a file that fails it (the trainer re-validates structure on top). `SKIP_VALIDATION = True` is
the explicit escape hatch (not recommended). `--eval` uses the **validation split**, *not*
`eval_holdout.jsonl` (that is for the final hand-review only).


In [23]:
import json, os

_STATE = '/kaggle/working/sahayak_state.json'

def state_load():
    return json.load(open(_STATE)) if os.path.exists(_STATE) else {}

def state_save(**kw):
    s = state_load(); s.update(kw)
    with open(_STATE, 'w') as f:
        json.dump(s, f, indent=1)
import glob, random, subprocess, sys

st = state_load()
FT = st.get('FT') or os.getcwd()
os.chdir(FT)

# ── knobs ─────────────────────────────────────────────────────────────────────
TRAIN_OVERRIDE = ''       # absolute path to force a specific training file
VAL_OVERRIDE = ''         # absolute path to force a specific val file
SKIP_VALIDATION = False   # explicit escape hatch ONLY — see the cell note above

# Data may come from an attached Kaggle dataset OR from the repo's own backend/finetune/data/
# (cloned by Cell 4) — an attached dataset always wins over the repo copy.
ROOTS = ['/kaggle/input', FT]

def find_all(pattern):
    hits = []
    for root in ROOTS:
        hits += glob.glob(os.path.join(root, '**', pattern), recursive=True)
    return sorted(set(hits), key=lambda p: (not p.startswith('/kaggle/input'), p))

def find_one(names):
    for n in names:
        hits = find_all(n)
        if hits:
            return hits[0]
    return None

all_jsonl = find_all('*.jsonl')

def by_keyword(key, exclude=()):
    named = [p for p in all_jsonl
             if key in os.path.basename(p).lower() and p not in exclude]
    return named[0] if named else None

TRAIN = TRAIN_OVERRIDE or find_one(['train.jsonl', 'all.jsonl']) or by_keyword('train')
# HOLD is resolved before VAL on purpose: 'val' is a substring of 'eval_holdout'.
HOLD  = find_one(['eval_holdout.jsonl', 'holdout.jsonl']) or by_keyword('hold', exclude={TRAIN})
VAL   = (VAL_OVERRIDE or find_one(['val.jsonl', 'valid.jsonl', 'validation.jsonl'])
         or by_keyword('val', exclude={TRAIN, HOLD}))

if not TRAIN:
    others = [p for p in all_jsonl if p not in {VAL, HOLD}
              and 'sample' not in os.path.basename(p).lower()]
    if len(others) == 1:
        TRAIN = others[0]
        print('no train.jsonl by name — using the only other JSONL found:', TRAIN)

if not TRAIN:
    print('No training JSONL found anywhere. Complete listing of /kaggle/input:')
    shown = 0
    for dirpath, _, files in os.walk('/kaggle/input'):
        for fn in files:
            if shown < 200:
                print('  ', os.path.join(dirpath, fn))
            shown += 1
    if shown == 0:
        print('   (EMPTY — no dataset is attached to this notebook)')
    elif shown > 200:
        print(f'   ... and {shown - 200} more files')
    data_dir = os.path.join(FT, 'data')
    print('repo data dir:', data_dir,
          '(exists)' if os.path.isdir(data_dir) else '(MISSING — re-run Cell 4)')
    raise SystemExit(
        'Could not find a training file. Fix ONE of these, then re-run this cell:\n'
        '  1. Attach your dataset: right side panel -> Input -> Add Input -> select your '
        'uploaded dataset, wait for it to mount, and confirm its files appear in the listing '
        'above (if the listing is EMPTY, nothing is attached);\n'
        '  2. or set TRAIN_OVERRIDE at the top of this cell to the exact file path;\n'
        '  3. or make sure the GitHub repo has backend/finetune/data/train.jsonl committed — Cell 4 '
        'clones it and this cell falls back to it automatically.')

print('data source:', 'attached Kaggle dataset'
      if TRAIN.startswith('/kaggle/input') else "the repo's backend/finetune/data/ (no dataset attached)")

if not VAL:
    lines = [l for l in open(TRAIN, encoding='utf-8') if l.strip()]
    if len(lines) >= 40:
        random.Random(3407).shuffle(lines)
        n_val = max(10, len(lines) // 20)  # ~5%
        os.makedirs('/kaggle/working/autosplit', exist_ok=True)
        VAL = '/kaggle/working/autosplit/val.jsonl'
        new_train = '/kaggle/working/autosplit/train.jsonl'
        with open(VAL, 'w', encoding='utf-8') as f:
            f.write(''.join(lines[:n_val]))
        with open(new_train, 'w', encoding='utf-8') as f:
            f.write(''.join(lines[n_val:]))
        TRAIN = new_train
        print(f'no val.jsonl found — auto-split {n_val} examples off the training file for eval.')
    else:
        print('[warn] no val.jsonl and the training file is tiny — training without eval loss.')

print('train  :', TRAIN)
print('val    :', VAL)
print('holdout:', HOLD)

have_validator = os.path.exists(os.path.join(FT, 'validate_dataset.py'))
if SKIP_VALIDATION or not have_validator:
    print('[warn] mechanical validation SKIPPED '
          + ('by the SKIP_VALIDATION flag.' if SKIP_VALIDATION else '(validator script missing).'))
else:
    failed = []
    for f in [p for p in (TRAIN, VAL) if p]:
        print(f'\n=== validating {f} ===')
        rc = subprocess.run([sys.executable, 'validate_dataset.py', f], cwd=FT).returncode
        if rc != 0:
            failed.append(f)
    if failed:
        raise SystemExit(
            'Validation FAILED for: ' + ', '.join(failed) + ' — fix the data (full error list '
            'above). Most common causes: system prompt not byte-identical to the spec prompt; '
            'near-duplicate user messages. If you knowingly accept the risk, set '
            'SKIP_VALIDATION = True in this cell and re-run (NOT recommended).')
    print('\ndataset validation passed.')

state_save(TRAIN=TRAIN, VAL=VAL, HOLD=HOLD)


data source: the repo's backend/finetune/data/ (no dataset attached)
train  : /kaggle/working/sankat-mochan/backend/finetune/data/train.jsonl
val    : /kaggle/working/sankat-mochan/backend/finetune/data/val.jsonl
holdout: /kaggle/working/sankat-mochan/backend/finetune/data/eval_holdout.jsonl

=== validating /kaggle/working/sankat-mochan/backend/finetune/data/train.jsonl ===
validated 1628 records from train.jsonl
OK — batch passes mechanical validation.

=== validating /kaggle/working/sankat-mochan/backend/finetune/data/val.jsonl ===
validated 172 records from val.jsonl
OK — batch passes mechanical validation.

dataset validation passed.


## Cell 6 — train E2B (QLoRA) with automatic OOM recovery

Best-accuracy settings from the project guide: **r=32 / α=32**, **lr 2e-4**, **3 epochs**, effective
batch **8** (1 × grad-accum 8), **seq-len 1024**. **Batch size 1 is deliberate on a T4**: float32
compute means the cross-entropy over Gemma's ~262k vocab materializes ~1 GB of logits **per
sequence** — batch 2 OOMs at step 0 (tested). Assistant-only loss masking is built into the trainer
(character-offset masking against the model's own chat template — the same template the on-device
runtime must use). Watch for a falling **eval** loss each epoch.

Robustness built in:
- **OOM auto-retry ladder** — seq-len `1024` → `768` → `512`, always batch 1 × grad-accum 8;
- **merged export auto-skipped when disk is low**, and if the merge fails *after* the adapters were
  saved, the run is treated as a **success** (re-merge later from the adapters);
- auth/gate, disk-full, and OOM failures are each reported with the exact fix.

_This runs the full training; expect ~1.5–2 h including the merged export._


In [24]:
import json, os

_STATE = '/kaggle/working/sahayak_state.json'

def state_load():
    return json.load(open(_STATE)) if os.path.exists(_STATE) else {}

def state_save(**kw):
    s = state_load(); s.update(kw)
    with open(_STATE, 'w') as f:
        json.dump(s, f, indent=1)
import shutil, subprocess, sys

st = state_load()
FT, TRAIN, VAL = st.get('FT'), st.get('TRAIN'), st.get('VAL')
BASE_MODEL = st.get('BASE_MODEL', 'google/gemma-4-E2B-it')
assert FT and TRAIN, 'State missing — run Cells 3–5 first (they are quick).'
os.chdir(FT)

OUT = '/kaggle/working/sahayak-e2b'

free_gb = shutil.disk_usage('/kaggle/working').free / 1e9
print(f'/kaggle/working free: {free_gb:.1f} GB')
EXPORT_MERGED = True
if free_gb < 12:
    print('[warn] low disk — skipping the merged export (full weights are ~10 GB). Adapters '
          'still save; re-merge later from them.')
    EXPORT_MERGED = False

def adapters_saved():
    return os.path.exists(os.path.join(OUT, 'adapter_config.json'))

def merged_saved():
    d = os.path.join(OUT, 'merged')
    return os.path.isdir(d) and any(f.endswith('.safetensors') for f in os.listdir(d))

def run_train(batch, accum, seq):
    cmd = [sys.executable, '-u', 'sahayak_finetune.py',
           '--train', TRAIN,
           '--model', BASE_MODEL,
           '--out', OUT,
           '--epochs', '3',
           '--lr', '2e-4',
           '--batch-size', str(batch),
           '--grad-accum', str(accum),
           '--lora-r', '32',
           '--lora-alpha', '32',
           '--max-seq-len', str(seq)]
    if VAL:
        cmd += ['--eval', VAL]
    if EXPORT_MERGED:
        cmd += ['--export-merged']
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
    print('\n$ ' + ' '.join(cmd) + '\n', flush=True)
    proc = subprocess.Popen(cmd, cwd=FT, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
        if len(tail) > 200:
            tail.pop(0)
    proc.wait()
    return proc.returncode, ''.join(tail)

# OOM-retry ladder: batch 1 is already the T4 floor (fp32 logits over the ~262k vocab are
# ~1 GB per sequence), so the ladder shrinks the sequence length instead.
LADDER = [(1, 8, 1024), (1, 8, 768), (1, 8, 512)]
done = False
for i, (b, a, s) in enumerate(LADDER, 1):
    rc, tail = run_train(b, a, s)
    if rc == 0:
        done = True
        break
    low = tail.lower()
    if adapters_saved():
        print('\n[warn] training + adapter save SUCCEEDED; a later step (the merged export — '
              'usually CPU RAM) failed. Continuing — the adapters are safe; re-merge later '
              'from them (see the export note below).')
        done = True
        break
    if 'out of memory' in low or 'outofmemoryerror' in low:
        if i < len(LADDER):
            nb, na, ns = LADDER[i]
            print(f'\n[retry] CUDA OOM at seq={s} -> retrying with seq={ns} '
                  f'(batch {nb} x grad-accum {na}, same effective batch) ...')
            shutil.rmtree(OUT, ignore_errors=True)
            continue
        raise SystemExit('CUDA OOM even at batch=1 / seq=512. Restart the session (frees VRAM) '
                         'and Run All from Cell 1 without running anything else on the GPU first.')
    if '401' in tail or '403' in tail or 'gatedrepo' in low or 'restricted' in low:
        raise SystemExit('Hugging Face refused the model download (auth/gated). Re-check the '
                         'HF_TOKEN secret and the accepted Gemma license, then re-run Cell 3 '
                         'and this cell.')
    if 'no space left on device' in low:
        raise SystemExit('Disk full in /kaggle/working. Delete old output folders, then re-run '
                         'this cell (the merged export auto-disables on low disk).')
    raise SystemExit('Training failed — the traceback is in the log above. Fix the cause and '
                     're-run this cell only; earlier cells keep their state.')

assert done and adapters_saved(), 'training did not produce adapters — see the log above'
state_save(OUT=OUT)
print('\ntraining complete -> ' + OUT)
print('merged export:', 'OK' if merged_saved() else 'not present (skipped or failed — see above)')


/kaggle/working free: 20.8 GB

$ /usr/bin/python3 -u sahayak_finetune.py --train /kaggle/working/sankat-mochan/backend/finetune/data/train.jsonl --model google/gemma-4-E2B-it --out /kaggle/working/sahayak-e2b --epochs 3 --lr 2e-4 --batch-size 1 --grad-accum 8 --lora-r 32 --lora-alpha 32 --max-seq-len 1024 --eval /kaggle/working/sankat-mochan/backend/finetune/data/val.jsonl --export-merged

── Sahayak fine-tune (transformers + PEFT) ───────────────────
 host      : Linux / x86_64 / py3.12.13
 device    : cuda (Tesla T4)
 precision : float32
──────────────────────────────────────────────────────────────
[data] train: 1628 records OK (/kaggle/working/sankat-mochan/backend/finetune/data/train.jsonl)
[data] eval: 172 records OK (/kaggle/working/sankat-mochan/backend/finetune/data/val.jsonl)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please us

> **On exports:** LoRA adapters (`OUT/`) are saved *before* the merge, so they are safe even if the
> merge step fails — Cell 6 detects that case and treats the run as a success. `OUT/merged/` holds
> the full merged weights — the **Qualcomm AI Hub** input. For a GGUF (llama.cpp CPU/GPU path), run
> llama.cpp's `convert_hf_to_gguf.py` (MIT-licensed) against `OUT/merged/` afterwards — GGUF
> conversion is deliberately out of this notebook.


## Cell 7 — quick qualitative check on the holdout _(optional)_

Loads the fine-tuned adapters (4-bit base + fp32 compute, same as training) and generates a few
held-out answers with Gemma 4's recommended sampling (`temp 1.0, top_p 0.95, top_k 64`). Falls back
to two built-in emergency prompts if no `eval_holdout.jsonl` exists. Eyeball packet format, refusal
calibration, length, and language. Optional — a failure here does not affect the trained artifacts.


In [29]:
import json, os

_STATE = '/kaggle/working/sahayak_state.json'

def state_load():
    return json.load(open(_STATE)) if os.path.exists(_STATE) else {}

def state_save(**kw):
    s = state_load(); s.update(kw)
    with open(_STATE, 'w') as f:
        json.dump(s, f, indent=1)
import subprocess, sys

st = state_load()
FT = st.get('FT')
OUT = st.get('OUT', '/kaggle/working/sahayak-e2b')
HOLD = st.get('HOLD')
BASE = st.get('BASE_MODEL', 'google/gemma-4-E2B-it')
assert FT and os.path.exists(os.path.join(OUT, 'adapter_config.json')), \
    'No trained adapters found — run Cell 6 first.'

# Best-effort: free VRAM held by any earlier in-kernel model so the subprocess gets the GPU.
try:
    import gc
    for _v in ('model', 'base', 'tok', 'ids', 'out'):
        if _v in globals():
            del globals()[_v]
    gc.collect()
    if 'torch' in sys.modules:
        sys.modules['torch'].cuda.empty_cache()
except Exception:
    pass

# ── disk-level health check in a FRESH interpreter; force-reinstall if corrupted ──
def _stack_ok():
    r = subprocess.run(
        [sys.executable, '-c',
         'import peft, transformers, bitsandbytes;'
         'from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig;'
         'print("transformers", transformers.__version__, "| peft", peft.__version__)'],
        capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode != 0:
        print(r.stderr[-1500:])
    return r.returncode == 0

if not _stack_ok():
    print('[repair] broken transformers/peft install detected -> clean forced reinstall ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                    '--no-cache-dir', 'transformers>=5.6,<6', 'peft>=0.16,<1',
                    'accelerate>=1.6,<2', 'bitsandbytes>=0.46,<1'], check=True)
    assert _stack_ok(), ('Still broken after the reinstall. Run -> Restart & clear cell '
                         'outputs, then re-run Cells 1 and 3, then this cell.')

# ── the actual check runs in a FRESH SUBPROCESS: immune to stale kernel imports, and it
# releases all GPU memory when it exits. The kernel never imports transformers/peft here.
SCRIPT = r"""
import json, os, sys

FT, OUT, BASE = sys.argv[1], sys.argv[2], sys.argv[3]
HOLD = sys.argv[4] if len(sys.argv) > 4 and sys.argv[4] != '-' else None
os.chdir(FT)
sys.path.insert(0, FT)

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sahayak_finetune import SYSTEM_PROMPT

tok = AutoTokenizer.from_pretrained(OUT)
base = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32,
    ),
    device_map={'': 0},
    attn_implementation='eager',
    token=os.environ.get('HF_TOKEN'),
)
model = PeftModel.from_pretrained(base, OUT)
model.eval()

prompts = []
if HOLD and os.path.exists(HOLD):
    with open(HOLD, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                o = json.loads(line)
            except json.JSONDecodeError:
                continue
            u = next((m.get('content') for m in o.get('messages', [])
                      if m.get('role') == 'user'), None)
            if u:
                prompts.append(u)
            if len(prompts) >= 3:
                break
if not prompts:
    prompts = [
        'My friend cut his leg badly and it will not stop bleeding. What do I do?',
        'Send an SOS: 3 people trapped near the old bridge, water rising, need a boat.',
    ]
    print('(no holdout prompts found - using built-in sample prompts)')

for u in prompts:
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': u}]
    # transformers 5.x returns a BatchEncoding dict here; ask for it explicitly and unpack —
    # passing the dict as input_ids crashes generate() with KeyError: 'shape'.
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True)
    enc = {k: v.to('cuda') for k, v in enc.items()}
    n_in = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=256, do_sample=True,
                             temperature=1.0, top_p=0.95, top_k=64)
    print('USER    :', u)
    print('SAHAYAK :', tok.decode(out[0][n_in:], skip_special_tokens=True))
    print('-' * 80)
"""

script_path = '/kaggle/working/qual_check.py'
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(SCRIPT)

cmd = [sys.executable, '-u', script_path, FT, OUT, BASE, HOLD or '-']
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
print('\n$ ' + ' '.join(cmd) + '\n', flush=True)
proc = subprocess.Popen(cmd, cwd=FT, env=env, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, \
    'qualitative check failed — see the log above (the trained artifacts are unaffected).'


transformers 5.13.1 | peft 0.19.1

$ /usr/bin/python3 -u /kaggle/working/qual_check.py /kaggle/working/sankat-mochan/backend/finetune /kaggle/working/sahayak-e2b google/gemma-4-E2B-it /kaggle/working/sankat-mochan/backend/finetune/data/eval_holdout.jsonl

/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 1951/1951 [00:04<00:00, 482.79it/s] 
USER    : a snake bit my brother on the ankle while we were wading out of the flooded field, his foot is swelling
SA

## Cell 8 — ship artifacts to a private HF repo

Uploads `adapters + merged/` to `<your-hf-user>/sahayak-e2b` (private). The repo owner is resolved
from your token, so no username is hard-coded. Re-derivable training checkpoints are excluded (they
are huge). The upload **retries 3×** and falls back off the `hf_transfer` fast path if it flakes.
A READ-only token is rejected up front with instructions.


In [26]:
import json, os

_STATE = '/kaggle/working/sahayak_state.json'

def state_load():
    return json.load(open(_STATE)) if os.path.exists(_STATE) else {}

def state_save(**kw):
    s = state_load(); s.update(kw)
    with open(_STATE, 'w') as f:
        json.dump(s, f, indent=1)
import time

st = state_load()
OUT = st.get('OUT', '/kaggle/working/sahayak-e2b')
assert os.path.isdir(OUT), 'No output folder — run Cell 6 first.'

from huggingface_hub import HfApi
token = os.environ.get('HF_TOKEN')
assert token, 'HF_TOKEN missing from the environment — re-run Cell 3.'
api = HfApi(token=token)
me = api.whoami()
role = (me.get('auth') or {}).get('accessToken', {}).get('role')
if role == 'read':
    raise SystemExit('Your HF token is READ-only; uploading needs a WRITE token. Create one at '
                     'huggingface.co/settings/tokens, update the Kaggle secret, then re-run '
                     'Cell 3 and this cell.')

user = me['name']
repo = f'{user}/sahayak-e2b'
api.create_repo(repo, private=True, exist_ok=True)

# training checkpoints are re-derivable and huge — keep them out of the upload
IGNORE = ['checkpoints/*', '**/checkpoints/*', '**/optimizer*', '**/rng_state*']

last_err = None
for attempt in range(1, 4):
    try:
        api.upload_folder(folder_path=OUT, repo_id=repo, ignore_patterns=IGNORE)
        print('uploaded -> https://huggingface.co/' + repo)
        break
    except Exception as e:
        last_err = e
        print(f'[warn] upload attempt {attempt}/3 failed: {e!r}')
        if attempt == 1:
            # hf_transfer occasionally flakes on Kaggle — fall back to the plain uploader
            os.environ.pop('HF_HUB_ENABLE_HF_TRANSFER', None)
            try:
                import huggingface_hub.constants as _c
                _c.HF_HUB_ENABLE_HF_TRANSFER = False
            except Exception:
                pass
        time.sleep(10 * attempt)
else:
    raise SystemExit(f'Upload failed 3 times: {last_err!r}. The artifacts are still safe in '
                     f'{OUT} — use Save Version so they persist in the notebook Output tab, '
                     'then upload from your machine later.')

print('pull locally:  huggingface-cli download', repo, '--local-dir ./sahayak-e2b')


uploaded -> https://huggingface.co/kesav2k04/sahayak-e2b
pull locally:  huggingface-cli download kesav2k04/sahayak-e2b --local-dir ./sahayak-e2b


## Artifacts & next step (NPU)

You get two artifacts — keep both:
- `merged/` (safetensors) → the **Qualcomm AI Hub** input for NPU compilation, and the input for a
  later GGUF conversion (llama.cpp `convert_hf_to_gguf.py`) for the app's llama.cpp CPU/GPU path.
- adapter files at the root (~100–200 MB) → re-merge later without retraining.

**Inference settings to bake into the app (must match at demo):** `temperature=1.0, top_p=0.95, top_k=64`,
the **model's own chat template** (saved with the tokenizer in `OUT/`), and the **byte-identical**
system prompt from the spec.

**NPU handoff (not on Kaggle):** AI Hub's LLM flow consumes the HF checkpoint (`merged/`), never GGUF —
quantize (AIMET, Linux/WSL + big-VRAM GPU) then export to QNN/Genie for `Snapdragon X Elite CRD`. See the
project's Kaggle guide §5 and `PLAN.md`.
